# Learnable Fusion vs. Regular Fusion: MRI Brain Tumor Preprocessing and ResUNet Baseline Comparison

This notebook implements and compares two preprocessing and multi-modal fusion approaches:
- **Regular mean fusion** (average of all modalities)
- **Learnable weighted fusion** (fusion weights are trained with the model)

Both pipelines are evaluated on a basic ResUNet architecture for a small number of epochs for mid-review reporting.

---


In [1]:
# Setup, imports, and config
import os, glob, random
import numpy as np
import SimpleITK as sitk
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt

# Config
DATA_ROOT = "/kaggle/input/brats2015/BRATS2015/training"
RANDOM_SEED = 42
PATCH_SIZE = (64,64,64)
N_CASES = 8
PATCHES_PER_CASE = 6
BATCH_SIZE = 2
EPOCHS = 2
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [2]:
# Utility: normalization, loader for all 4 modalities, z-score per channel
def zscore_norm(img):
    mask = img > 0
    if mask.sum() == 0:
        return (img - img.mean()) / (img.std() + 1e-8)
    else:
        mean = img[mask].mean()
        std = img[mask].std()
        return (img - mean) / (std + 1e-8)

def load_mha_case(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.mha")))
    t1 = next(f for f in files if "t1." in f.lower() and not "t1c" in f.lower())
    t1ce = next(f for f in files if "t1c" in f.lower())
    t2 = next(f for f in files if "t2." in f.lower())
    flair = next(f for f in files if "flair" in f.lower())
    seg = next(f for f in files if "ot." in f.lower())
    vols = []
    for f in [t1, t2, t1ce, flair]:
        img = sitk.GetArrayFromImage(sitk.ReadImage(f)).astype(np.float32)
        img = zscore_norm(img)
        vols.append(img)
    vol = np.stack(vols, axis=0)  # [4, D, H, W]
    seg = sitk.GetArrayFromImage(sitk.ReadImage(seg)).astype(np.int16)
    return vol, seg



In [3]:
# Dataset v1: REGULAR MEAN FUSION (for classic pipeline)
class RegularFusionPatchDataset(Dataset):
    def __init__(self, data_root, n_cases=N_CASES, patch=PATCH_SIZE, patches_per_case=PATCHES_PER_CASE):
        folders = sorted(glob.glob(os.path.join(data_root, "*/*/")))
        random.shuffle(folders)
        selected = folders[:n_cases]
        self.patch = patch
        self.patches_per_case = patches_per_case
        self.volumes, self.segs = [], []
        print(f"Loading {len(selected)} cases...")
        for case in selected:
            try:
                v, l = load_mha_case(case)
                self.volumes.append(v)
                self.segs.append(l)
            except Exception as e:
                print(f"Skipping {case} due to {e}")
        print("Done loading.")
    def __len__(self):
        return len(self.volumes) * self.patches_per_case
    def __getitem__(self, idx):
        cidx = idx // self.patches_per_case
        img4 = self.volumes[cidx]
        seg = self.segs[cidx]
        D,H,W = img4.shape[1:]
        pd,ph,pw=self.patch
        z0 = random.randint(0, D-pd)
        y0 = random.randint(0, H-ph)
        x0 = random.randint(0, W-pw)
        patch4 = img4[:,z0:z0+pd,y0:y0+ph,x0:x0+pw]    # [4, pd, ph, pw]
        fused = patch4.mean(axis=0, keepdims=True)      # [1, pd, ph, pw]
        label = seg[z0:z0+pd,y0:y0+ph,x0:x0+pw]
        fused = torch.from_numpy(fused).float()
        label = torch.from_numpy(label).long()
        return fused, label



In [4]:
# Dataset v2: LEARNABLE FUSION (delivers all 4 modalities, fusion in the model)
class LearnableFusionPatchDataset(Dataset):
    def __init__(self, data_root, n_cases=N_CASES, patch=PATCH_SIZE, patches_per_case=PATCHES_PER_CASE):
        folders = sorted(glob.glob(os.path.join(data_root, "*/*/")))
        random.shuffle(folders)
        selected = folders[:n_cases]
        self.patch = patch
        self.patches_per_case = patches_per_case
        self.volumes, self.segs = [], []
        print(f"Loading {len(selected)} cases...")
        for case in selected:
            try:
                v, l = load_mha_case(case)
                self.volumes.append(v)
                self.segs.append(l)
            except Exception as e:
                print(f"Skipping {case} due to {e}")
        print("Done loading.")
    def __len__(self):
        return len(self.volumes) * self.patches_per_case
    def __getitem__(self, idx):
        cidx = idx // self.patches_per_case
        img4 = self.volumes[cidx]
        seg = self.segs[cidx]
        D,H,W = img4.shape[1:]
        pd,ph,pw=self.patch
        z0 = random.randint(0, D-pd)
        y0 = random.randint(0, H-ph)
        x0 = random.randint(0, W-pw)
        patch4 = img4[:,z0:z0+pd,y0:y0+ph,x0:x0+pw]    # [4, pd, ph, pw]
        label = seg[z0:z0+pd,y0:y0+ph,x0:x0+pw]
        patch4 = torch.from_numpy(patch4).float()
        label = torch.from_numpy(label).long()
        return patch4, label


In [5]:
# -- Model Definitions --
class ResidualBlock3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.proj = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        identity = self.proj(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv3d(in_ch, in_ch, 3, stride=2, padding=1)
        self.rb = ResidualBlock3d(in_ch, out_ch)
    def forward(self, x):
        x = self.down(x)
        return self.rb(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, skip_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.rb = ResidualBlock3d(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        ds = [skip.size(d) - x.size(d) for d in range(2,5)]
        if any(ds):
            x = nn.functional.pad(x, [0, ds[2], 0, ds[1], 0, ds[0]])
        x = torch.cat((x, skip), 1)
        return self.rb(x)

class SimpleResUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=5, base=8):
        super().__init__()
        self.stem = ResidualBlock3d(in_ch, base)
        self.down1 = DownBlock(base, base*2)
        self.down2 = DownBlock(base*2, base*4)
        self.down3 = DownBlock(base*4, base*8)
        self.down4 = DownBlock(base*8, base*8)
        self.up1 = UpBlock(base*8, base*4, base*8)
        self.up2 = UpBlock(base*4, base*2, base*4)
        self.up3 = UpBlock(base*2, base, base*2)
        self.up4 = UpBlock(base, base, base)
        self.head = nn.Conv3d(base, out_ch, 1)
    def forward(self, x):
        s1 = self.stem(x)
        d1 = self.down1(s1)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        u1 = self.up1(d4, d3)
        u2 = self.up2(u1, d2)
        u3 = self.up3(u2, d1)
        u4 = self.up4(u3, s1)
        return self.head(u4)

class LearnableFusionResUNet3D(SimpleResUNet3D):
    def __init__(self, in_ch=4, out_ch=5, base=8):
        super().__init__(in_ch=1, out_ch=out_ch, base=base)
        self.fusion_weights = nn.Parameter(torch.ones(in_ch))
    def forward(self, x):
        fw = torch.softmax(self.fusion_weights, dim=0)
        x_fused = (x * fw.view(1,4,1,1,1)).sum(dim=1, keepdim=True)
        return super().forward(x_fused)



In [6]:
# --- Simple Dice Loss and Score Function ---
def dice_loss(pred, target, num_classes=5):
    pred_soft = torch.softmax(pred, dim=1)
    onehot = torch.nn.functional.one_hot(target, num_classes=num_classes).permute(0,4,1,2,3).float()
    intersect = (pred_soft * onehot).sum(dim=(2,3,4))
    denom = pred_soft.sum(dim=(2,3,4)) + onehot.sum(dim=(2,3,4))
    loss = 1 - (2*intersect+1) / (denom+1)
    return loss.mean()
def dice_score(pred, target, num_classes=5):
    pred = torch.argmax(pred, dim=1)
    dices = []
    for c in range(num_classes):
        p = (pred == c)
        t = (target == c)
        inter = (p & t).sum().item()
        den = p.sum().item() + t.sum().item()
        dices.append((2*inter+1e-4)/(den+1e-4) if den > 0 else 1.0)
    return np.mean(dices)



In [7]:
# Pipeline: Train & Compare Regular Fusion and Learnable Fusion

# Dataloaders
loader1 = DataLoader(RegularFusionPatchDataset(DATA_ROOT), batch_size=2, shuffle=True)
loader2 = DataLoader(LearnableFusionPatchDataset(DATA_ROOT), batch_size=2, shuffle=True)

# Models
model1 = SimpleResUNet3D(in_ch=1, out_ch=5, base=8).to(DEVICE)
model2 = LearnableFusionResUNet3D(in_ch=4, out_ch=5, base=8).to(DEVICE)
opt1 = torch.optim.AdamW(model1.parameters(), lr=1e-3)
opt2 = torch.optim.AdamW(model2.parameters(), lr=1e-3)

# Training (minimal: 2 epochs)
for name, loader, model, opt in [
    ("RegFusion", loader1, model1, opt1),
    ("LearnFusion", loader2, model2, opt2)
]:
    print(f"\nTraining {name} pipeline...")
    for epoch in range(EPOCHS):
        model.train(); total_loss, total_dice = [], []
        for x, y in tqdm(loader, desc=f"{name} Epoch {epoch+1}"):
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = dice_loss(out, y)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss.append(loss.item())
            total_dice.append(dice_score(out.detach().cpu(), y.cpu()))
        print(f"{name} Epoch {epoch+1}: loss={np.mean(total_loss):.4f} | dice={np.mean(total_dice):.4f}")
        if name=="LearnFusion":
            print("Fusion Weights (softmaxed):", model.fusion_weights.softmax(dim=0).detach().cpu().numpy())


Loading 8 cases...
Done loading.
Loading 8 cases...
Done loading.

Training RegFusion pipeline...


RegFusion Epoch 1: 100%|██████████| 24/24 [00:06<00:00,  3.82it/s]


RegFusion Epoch 1: loss=0.9245 | dice=0.1073


RegFusion Epoch 2: 100%|██████████| 24/24 [00:05<00:00,  4.70it/s]


RegFusion Epoch 2: loss=0.8951 | dice=0.1824

Training LearnFusion pipeline...


LearnFusion Epoch 1: 100%|██████████| 24/24 [00:05<00:00,  4.69it/s]


LearnFusion Epoch 1: loss=0.9045 | dice=0.1657
Fusion Weights (softmaxed): [0.24558638 0.25380307 0.24662681 0.2539838 ]


LearnFusion Epoch 2: 100%|██████████| 24/24 [00:05<00:00,  4.72it/s]

LearnFusion Epoch 2: loss=0.8783 | dice=0.3262
Fusion Weights (softmaxed): [0.24064718 0.25809494 0.2418138  0.2594441 ]
